In [94]:
from sklearn.datasets import load_breast_cancer

RANDOM_STATE = 42

In [95]:
(X, y) = load_breast_cancer(return_X_y = True)

In [96]:
import numpy as np
np.unique(y, return_counts=True)

(array([0, 1]), array([212, 357]))

In [97]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [106]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import optuna

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

def objective(trial):
    activation = trial.suggest_categorical(
        "activation", ["identity", "logistic", "tanh", "relu"]
    )

    alpha = trial.suggest_float(
        "alpha", 1e-5, 10, log=True
    )

    n_layers = trial.suggest_int("n_layers", 1, 3)

    hidden_layer_sizes = tuple(
        trial.suggest_int(f"n_units_l{i}", 30, 128, log=True)
        for i in range(n_layers)
    )

    learning_rate_init = trial.suggest_float(
        "learning_rate_init", 1e-5, 1, log=True
    )

    model = MLPClassifier(
        activation=activation,
        alpha=alpha,
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate_init,
        max_iter=10000,
        early_stopping=True,
        random_state=RANDOM_STATE
    )

    # Оценка на 5 фолдах
    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )

    return scores.mean()

# Запуск оптимизации
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

study.optimize(objective, n_trials=1000)

print("Лучший SCORE:", study.best_value)
print("Лучшие параметры:")
print(study.best_params)


[I 2026-10-13 13:08:13,471] A new study created in memory with name: no-name-ac14189d-14eb-4413-8c90-32eca926f186
[I 2026-10-13 13:08:13,513] Trial 0 finished with value: 0.9649679584986266 and parameters: {'activation': 'logistic', 'alpha': 8.632008168602535e-05, 'n_layers': 1, 'n_units_l0': 32, 'learning_rate_init': 0.21423021757741043}. Best is trial 0 with value: 0.9649679584986266.
[I 2026-10-13 13:08:13,588] Trial 1 finished with value: 0.7845376222697442 and parameters: {'activation': 'relu', 'alpha': 0.9877700294007907, 'n_layers': 1, 'n_units_l0': 39, 'learning_rate_init': 8.260808399079601e-05}. Best is trial 0 with value: 0.9649679584986266.
[I 2026-10-13 13:08:13,629] Trial 2 finished with value: 0.8913519539262735 and parameters: {'activation': 'logistic', 'alpha': 0.046894009635376835, 'n_layers': 1, 'n_units_l0': 45, 'learning_rate_init': 0.0006789053271698484}. Best is trial 0 with value: 0.9649679584986266.
[I 2026-10-13 13:08:13,705] Trial 3 finished with value: 0.509

Лучший SCORE: 0.9877947798341165
Лучшие параметры:
{'activation': 'tanh', 'alpha': 0.0013290064970307208, 'n_layers': 3, 'n_units_l0': 50, 'n_units_l1': 51, 'n_units_l2': 128, 'learning_rate_init': 0.03481648468897868}


In [108]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

clf = MLPClassifier(
    solver="lbfgs",
    activation="tanh",
    alpha=0.0013290064970307208,
    hidden_layer_sizes=[50, 51, 128],
    learning_rate_init=0.03481648468897868,
    max_iter=10000,
    random_state=RANDOM_STATE,
)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.932     0.976     0.953        42
           1      0.986     0.958     0.972        72

    accuracy                          0.965       114
   macro avg      0.959     0.967     0.963       114
weighted avg      0.966     0.965     0.965       114

